# Introduction to Tools in LangChain

In LangChain, **Tools** allow LLMs to interact with external systems (like databases, APIs, web search, or local code execution). 

A Tool in LangChain consists of two parts:
1. **The Schema:** A JSON schema specifying the tool's name, description, and input parameters (arguments). The model uses this schema to decide when and how to call the tool.
2. **The Executable Function:** A Python function or coroutine that actually performs the action (e.g., queries a database or fetches weather data).

## 1. Initializing the Model

First, we load the environment variables from the `.env` file and initialize our model using the unified interface `init_chat_model`.

In [10]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# Qwen model (using OpenAI client compatible configuration)
model = init_chat_model(
    model=os.getenv("QWEN_MODEL", "qwen-turbo"),
    model_provider="openai",
    api_key=os.getenv("QWEN_API_KEY"),
    base_url=os.getenv("QWEN_BASE_URL")
)

# Groq model (Commented Out)
# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
# model = init_chat_model("groq:qwen/qwen3-32b")

response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots talk because they have the ability to **learn and mimic human speech**, which is a result of their **complex vocal anatomy** and **highly developed brains**. Here\'s a breakdown of why parrots can "talk":\n\n### 1. **Vocal Anatomy**\n- Parrots have a specialized organ called the **syrinx**, which is their voice box. Unlike humans, who use the larynx (in the throat), parrots produce sounds using the syrinx, located at the base of the trachea.\n- This allows them to create a wide range of sounds, including human-like speech.\n\n### 2. **High Intelligence**\n- Parrots are among the most intelligent birds. Species like African Grey Parrots and Amazon Parrots have been shown to understand concepts such as color, shape, and even basic numbers.\n- Their intelligence helps them associate words with meanings, not just repeat sounds.\n\n### 3. **Social Nature**\n- In the wild, parrots are highly social animals that use a variety of calls to communicate with each other.

## 2. Defining and Binding Tools

To define a tool in LangChain, we use the `@tool` decorator on a Python function.

> **Important:** The function's **docstring** (`"""Get the weather at a location"""`) and the **argument type hints** (`location:str`) are critical! LangChain uses them to generate the tool's schema, which is sent to the LLM so it understands what the tool does and what inputs it requires.

Once defined, we bind the tool to the model using `model.bind_tools([get_weather])`. This produces a new wrapper (`model_with_tools`) that is aware of the tool schema.

In [11]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

## 3. Invoking the Model to Request a Tool Call

When we invoke `model_with_tools`, the LLM **does not run the tool**. Instead, the LLM reads the user prompt, recognizes that it needs external weather information, and returns a response containing a **tool call request** in `response.tool_calls` specifying the function name and arguments.

In [12]:
response = model_with_tools.invoke("What's the weather like in Navsari, Gujarat ?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 162, 'total_tokens': 185, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-458a8864-6558-9b0d-9ddb-7dec74f6fc95', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f4380-5be8-7980-bad8-97cda1ea90b5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Navsari, Gujarat'}, 'id': 'call_2eed5579589b4471a570a5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 162, 'output_tokens': 23, 'total_tokens': 185, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
Tool: get_weather
Args: {'location': 'Navsari, Gujarat'}


## 4. The Tool Execution Loop

To actually get the answer, we must execute the tool loop. This loop involves three steps:

1. **Step 1: Model generates tool calls.** We invoke the model with the user's prompt. The model returns an `AIMessage` with `tool_calls` specifying which tool to execute and with what arguments.
2. **Step 2: Execute the tool.** We extract the tool call metadata, execute the Python function using `.invoke(tool_call)` (which automatically runs the function with the model's generated arguments), and append the result to the history as a `ToolMessage`.
3. **Step 3: Final Answer.** We pass the entire conversation history (User message + AIMessage tool call + ToolMessage result) back to the model. The model reads the tool output and writes the final natural language answer.

In [13]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)
# "The current weather in Boston is 72°F and sunny."

# Display the list of messages in the conversation
messages

It's sunny in Boston.


[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 157, 'total_tokens': 176, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-ce9e0b05-a246-98f4-99fb-3dbdf5bb27d8', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f4380-5da9-7873-99c3-9346af3f9cc8-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_d380bd3867a44e03978ef6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 19, 'total_tokens': 176, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='call_d380bd3867a44e03978ef6')]